In [4]:
# !pip install nba_api

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from nba_api.stats.endpoints import leaguedashteamstats
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import confusion_matrix, classification_report
import warnings

warnings.filterwarnings('ignore')

os.makedirs('outputs', exist_ok=True)

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")


def fetch_team_data(seasons=None):
    """Fetch NBA team data from multiple seasons."""
    if seasons is None:
        seasons = [
            '2019-20',
            '2020-21',
            '2021-22',
            '2022-23',
            '2023-24',
            '2024-25'
        ]

    print(f"Fetching NBA team data for {len(seasons)} seasons...")

    all_dfs = []

    for season in seasons:
        print(f"  -> Fetching {season}")
        team_stats = leaguedashteamstats.LeagueDashTeamStats(
            season=season,
            season_type_all_star='Regular Season',
            measure_type_detailed_defense='Advanced'
        )
        df = team_stats.get_data_frames()[0]
        df['SEASON'] = season  # 👈 track season
        all_dfs.append(df)

    combined_df = pd.concat(all_dfs, ignore_index=True)

    print(f"Total rows collected: {len(combined_df)}")

    return combined_df


def prepare_data(df):
    """Prepare features and create target variable."""
    features = [
        'OFF_RATING',
        'DEF_RATING',
        'NET_RATING',
        'PACE',
        'TS_PCT',
        'EFG_PCT',
        'AST_RATIO',
        'REB_PCT',
        'TM_TOV_PCT',
        'PIE'
    ]

    df_clean = df[['TEAM_NAME'] + features + ['W_PCT']].dropna()
    df_clean['WIN_TIER'] = pd.qcut(
        df_clean['W_PCT'],
        3,
        labels=['Low', 'Mid', 'High'],
        duplicates='drop'
    )

    X = df_clean[features].copy()
    y = df_clean['WIN_TIER']
    team_names = df_clean['TEAM_NAME']

    print(f"Data shape: {X.shape}")
    print(f"Target distribution:\n{y.value_counts().sort_index()}")

    return X, y, team_names, features


def plot_labeled_confusion_matrix(y_true, y_pred, class_names, title, filename):
    """Plot and save a labeled confusion matrix."""
    cm = confusion_matrix(y_true, y_pred, labels=class_names)

    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Greens',
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.close()
    print(f"Saved: {filename}")


def plot_decision_tree(tree, feature_names, class_names, title, filename, max_depth=3):
    """Plot and save a decision tree visualization."""
    plt.figure(figsize=(20, 10))
    plot_tree(
        tree,
        feature_names=feature_names,
        class_names=class_names,
        filled=True,
        rounded=True,
        fontsize=10,
        max_depth=max_depth
    )
    plt.title(title, fontsize=16, pad=20)
    plt.savefig(filename, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved: {filename}")


def main():
    df = fetch_team_data()
    X, y, team_names, feature_names = prepare_data(df)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    class_names = y.unique().tolist()

    print("\n--- Full Decision Tree (unlimited depth) ---")
    clf_full = DecisionTreeClassifier(random_state=42)
    clf_full.fit(X_train, y_train)

    y_pred = clf_full.predict(X_test)

    print(classification_report(y_test, y_pred, target_names=class_names))

    plot_labeled_confusion_matrix(
        y_test, y_pred, class_names,
        'Decision Tree - Confusion Matrix',
        'outputs/decision_tree_confusion.png'
    )

    plot_decision_tree(
        clf_full, feature_names, class_names,
        'Decision Tree - Full Structure (max_depth=4 shown)',
        'outputs/decision_tree_full.png',
        max_depth=4
    )

    print("\n--- Tree 1: max_features=3 ---")
    clf_tree1 = DecisionTreeClassifier(
        max_features=3,
        random_state=42
    )
    clf_tree1.fit(X_train, y_train)
    plot_decision_tree(
        clf_tree1, feature_names, class_names,
        'Decision Tree 1 - max_features=3',
        'outputs/decision_tree_1_maxfeat3.png',
        max_depth=3
    )
    print(f"Root node feature: {feature_names[clf_tree1.tree_.feature[0]]}")

    print("\n--- Tree 2: max_features=5 ---")
    clf_tree2 = DecisionTreeClassifier(
        max_features=5,
        random_state=42
    )
    clf_tree2.fit(X_train, y_train)
    plot_decision_tree(
        clf_tree2, feature_names, class_names,
        'Decision Tree 2 - max_features=5',
        'outputs/decision_tree_2_maxfeat5.png',
        max_depth=3
    )
    print(f"Root node feature: {feature_names[clf_tree2.tree_.feature[0]]}")

    print("\n--- Tree 3: max_features='sqrt' ---")
    clf_tree3 = DecisionTreeClassifier(
        max_features='sqrt',
        random_state=42
    )
    clf_tree3.fit(X_train, y_train)
    plot_decision_tree(
        clf_tree3, feature_names, class_names,
        'Decision Tree 3 - max_features=sqrt',
        'outputs/decision_tree_3_maxfeatsqrt.png',
        max_depth=3
    )
    
    y_pred1 = clf_tree1.predict(X_test)
    plot_labeled_confusion_matrix(
        y_test, y_pred1, class_names,
        'Decision Tree 1 - Confusion Matrix (max_features=3)',
        'outputs/decision_tree_1_confusion.png'
    )
    
    y_pred2 = clf_tree2.predict(X_test)
    plot_labeled_confusion_matrix(
        y_test, y_pred2, class_names,
        'Decision Tree 2 - Confusion Matrix (max_features=5)',
        'outputs/decision_tree_2_confusion.png'
    )
    
    y_pred3 = clf_tree3.predict(X_test)
    plot_labeled_confusion_matrix(
        y_test, y_pred3, class_names,
        'Decision Tree 3 - Confusion Matrix (max_features=sqrt)',
        'outputs/decision_tree_3_confusion.png'
    )
    print(f"Root node feature: {feature_names[clf_tree3.tree_.feature[0]]}")

    print("\n=== Summary ===")
    print("All outputs saved to /outputs folder:")
    print("  - decision_tree_confusion.png (full tree confusion matrix)")
    print("  - decision_tree_full.png (full tree structure)")
    print("  - decision_tree_1_maxfeat3.png (tree with max_features=3)")
    print("  - decision_tree_2_maxfeat5.png (tree with max_features=5)")
    print("  - decision_tree_3_maxfeatsqrt.png (tree with max_features=sqrt)")


if __name__ == "__main__":
    main()


Fetching NBA team data for 6 seasons...
  -> Fetching 2019-20
  -> Fetching 2020-21
  -> Fetching 2021-22
  -> Fetching 2022-23
  -> Fetching 2023-24
  -> Fetching 2024-25
Total rows collected: 180
Data shape: (180, 10)
Target distribution:
WIN_TIER
Low     60
Mid     66
High    54
Name: count, dtype: int64

--- Full Decision Tree (unlimited depth) ---
              precision    recall  f1-score   support

         Low       0.90      0.82      0.86        11
        High       0.92      0.92      0.92        12
         Mid       0.79      0.85      0.81        13

    accuracy                           0.86        36
   macro avg       0.87      0.86      0.86        36
weighted avg       0.86      0.86      0.86        36

Saved: outputs/decision_tree_confusion.png
Saved: outputs/decision_tree_full.png

--- Tree 1: max_features=3 ---
Saved: outputs/decision_tree_1_maxfeat3.png
Root node feature: NET_RATING

--- Tree 2: max_features=5 ---
Saved: outputs/decision_tree_2_maxfeat5.png
R